# RAG Prototype — Ingestion → Markdown Conversion → Section Splitting → Chunking

This builds on the ingestion/structuring notebook, with the changes we talked through:

- **PDF extraction is now line-level** (like DOCX is paragraph-level and XLSX is row-level), so every format
  gives one record per atomic unit, each carrying whatever structural signal that format actually has
  (`style` for DOCX, `font_size`/`is_bold` for PDF).
- **DOCX and PDF are converted to Markdown** using those signals (`# `/`## ` for headings), instead of each
  format having its own hand-written section-builder. One `MarkdownHeaderTextSplitter` now does the section
  splitting for both formats — this is the "why do docx and pdf need separate structuring code" problem
  actually going away, rather than just being papered over.
- **XLSX skips section-splitting and semantic chunking entirely** — a row is already one complete, atomic
  semantic unit, so it goes straight from record to chunk.
- **DOCX/PDF sections get semantically chunked** (embedding similarity between sentences, splitting where
  meaning shifts) instead of blind fixed-size windows — with the fixed-size sliding-window chunker from the
  previous notebook kept as an automatic fallback for short sections, oversized semantic chunks, or if the
  embedding model isn't available in this environment.
- A verification cell sits after ingestion, after markdown conversion, and after chunking — so you can see
  exactly what each step actually produced before moving on to the next.

Deduplication is *not* implemented here — that's the next notebook, once you're ready to walk through it.

## 1. Imports

In [2]:
import json
import logging
import re
from dataclasses import dataclass, field
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import pymupdf
from docx import Document
from openpyxl import load_workbook
from langchain_text_splitters import MarkdownHeaderTextSplitter

try:
    from sentence_transformers import SentenceTransformer
    _SENTENCE_TRANSFORMERS_AVAILABLE = True
except ImportError:
    _SENTENCE_TRANSFORMERS_AVAILABLE = False

## 2. Configuration & Logging

New fields since the last notebook: `markdown_headers` (how many heading levels the splitter recognizes),
and the semantic-chunking knobs (`semantic_chunk_percentile` controls how aggressively it splits — higher =
fewer, bigger chunks; `semantic_chunk_min_sentences` skips semantic chunking for sections too short for it to
be meaningful; `semantic_chunk_max_chars` is the hard ceiling a chunk still can't exceed even after semantic
grouping, enforced by the fixed-size chunker as a fallback).

In [ ]:
@dataclass
class Config:
    base_dir: Path = field(default_factory=Path.cwd)
    data_subdir: str = "data"
    raw_subdir: str = "raw"                # data/raw       — untouched source files
    processed_subdir: str = "processed"    # data/processed — ingestion + chunking output
    output_subdir: str = "outputs"
    log_subdir: str = "logs"

    supported_extensions: tuple = (".pdf", ".docx", ".xlsx")

    short_page_word_threshold: int = 20

    # A line that repeats across at least this fraction of a PDF's pages is
    # treated as a running header/footer and stripped.
    pdf_boilerplate_min_repeat_ratio: float = 0.4

    # PDF heading detection: a line's font_size divided by the document's body
    # font size, compared against these ratios.
    pdf_heading_size_ratio_h1: float = 1.4
    pdf_heading_size_ratio_h2: float = 1.15

    # An xlsx column whose non-empty values average fewer words than this is
    # treated as an identifier/category rather than free text.
    xlsx_semantic_min_avg_words: float = 3.0
    xlsx_id_like_names: frozenset = frozenset({
        "id", "row_id", "index", "rank", "ranking", "serial", "sr_no", "s_no",
    })

    # Markdown heading levels the section splitter recognizes (#, ##, ###, ####).
    markdown_headers: tuple = (("#", "h1"), ("##", "h2"), ("###", "h3"), ("####", "h4"))

    # Semantic chunking (docx/pdf sections only — xlsx rows are chunked directly)
    embedding_model_name: str = "all-MiniLM-L6-v2"
    semantic_chunk_percentile: float = 95.0     # higher = fewer, larger chunks
    semantic_chunk_min_sentences: int = 4       # below this, just keep the section as one chunk
    semantic_chunk_max_chars: int = 1500        # hard ceiling; oversized chunks get sliding-window split
    fixed_chunk_overlap: int = 200

    @property
    def data_dir(self) -> Path:
        return self.base_dir / self.data_subdir

    @property
    def raw_dir(self) -> Path:
        return self.data_dir / self.raw_subdir

    @property
    def processed_dir(self) -> Path:
        return self.data_dir / self.processed_subdir

    @property
    def output_dir(self) -> Path:
        return self.base_dir / self.output_subdir

    @property
    def log_dir(self) -> Path:
        return self.base_dir / self.log_subdir

    def ensure_dirs(self):
        for directory in (self.raw_dir, self.processed_dir, self.output_dir, self.log_dir):
            directory.mkdir(exist_ok=True, parents=True)


CONFIG = Config()
CONFIG.ensure_dirs()

print(f"Raw directory:       {CONFIG.raw_dir}")
print(f"Processed directory: {CONFIG.processed_dir}")
print(f"Output directory:    {CONFIG.output_dir}")
print(f"Log directory:       {CONFIG.log_dir}")
print(f"sentence-transformers available: {_SENTENCE_TRANSFORMERS_AVAILABLE}")

**One-time manual step:** source files go in `data/raw/`. Everything below reads from `raw_dir` and
writes to `processed_dir`.

In [ ]:
logger = logging.getLogger("rag_ingestion")
logger.setLevel(logging.INFO)
logger.handlers.clear()

_formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

_console_handler = logging.StreamHandler()
_console_handler.setFormatter(_formatter)
logger.addHandler(_console_handler)

_file_handler = logging.FileHandler(CONFIG.log_dir / "ingestion.log")
_file_handler.setFormatter(_formatter)
logger.addHandler(_file_handler)

logger.info("Logger initialized.")

## 3. Data Inventory

In [ ]:
def list_data_files(config: Config = CONFIG) -> list[Path]:
    files = sorted(config.raw_dir.iterdir())

    logger.info(f"Files in raw directory: {len(files)}")
    for file in files:
        size_kb = file.stat().st_size / 1024
        logger.info(f"- {file.name} | {file.suffix or 'no ext'} | {size_kb:.2f} KB")

    return files


data_files = list_data_files()

## 4. Document Extraction

### 4.1 Shared helpers

In [ ]:
def clean_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def compute_stats(text: str) -> dict:
    return {
        "char_count": len(text),
        "word_count": len(text.split()),
        "line_count": len(text.splitlines()),
    }


def make_record(document: str, source_type: str, location: str, text: str, metadata: dict | None = None) -> dict:
    cleaned = clean_text(text)
    record = {
        "document": document,
        "source_type": source_type,
        "location": location,
        "text": cleaned,
        "metadata": metadata or {},
    }
    record.update(compute_stats(cleaned))
    return record

### 4.2 PDF — line-level, with font metadata

Changed from the last notebook: PDF used to be one record per **page**. It's now one record per **line**,
the same granularity as DOCX's per-paragraph records — because line-level is what heading detection (font
size, boldness) needs, and there's no reason to keep two different granularities around once one of them is
strictly more useful. `page.get_text("dict")` gives per-line font info that plain `page.get_text()` doesn't.

In [ ]:
def find_boilerplate_lines(page_texts: list[str], config: Config = CONFIG) -> set[str]:
    """Lines repeating across a large fraction of a document's pages — running
    headers/footers rather than content."""
    if len(page_texts) < 3:
        return set()

    line_counts = Counter()
    for text in page_texts:
        unique_lines_on_page = {line.strip() for line in text.splitlines() if line.strip()}
        line_counts.update(unique_lines_on_page)

    threshold = max(2, int(len(page_texts) * config.pdf_boilerplate_min_repeat_ratio))
    return {line for line, count in line_counts.items() if count >= threshold}


def extract_pdf(file_path: Path, config: Config = CONFIG) -> list[dict]:
    with pymupdf.open(file_path) as doc:
        page_texts = [page.get_text() for page in doc]
        boilerplate = find_boilerplate_lines(page_texts, config)
        if boilerplate:
            logger.info(f"{file_path.name}: stripping {len(boilerplate)} repeated header/footer line(s)")

        documents = []
        for page_number, page in enumerate(doc, start=1):
            for line_number, block in enumerate(page.get_text("dict")["blocks"], start=1):
                for line in block.get("lines", []):
                    spans = line.get("spans", [])
                    if not spans:
                        continue

                    raw_text = "".join(span["text"] for span in spans).strip()
                    if not raw_text or raw_text in boilerplate:
                        continue

                    documents.append(
                        make_record(
                            document=file_path.name,
                            source_type="pdf",
                            location=f"page_{page_number}_line_{line_number}",
                            text=raw_text,
                            metadata={
                                "page": page_number,
                                "font_size": round(max(s["size"] for s in spans), 1),
                                "is_bold": any("bold" in s["font"].lower() for s in spans),
                            },
                        )
                    )

    return documents

### 4.3 DOCX

In [ ]:
def extract_docx(file_path: Path) -> list[dict]:
    documents = []

    doc = Document(file_path)

    for index, paragraph in enumerate(doc.paragraphs, start=1):
        text = paragraph.text.strip()

        if text:
            documents.append(
                make_record(
                    document=file_path.name,
                    source_type="docx",
                    location=f"paragraph_{index}",
                    text=text,
                    metadata={"style": paragraph.style.name},
                )
            )

    return documents

### 4.4 XLSX — column classification

Unchanged from before: `semantic` columns get joined into `text`; `metadata` columns (ids, categories) stay
attached as structured metadata instead of being mashed into the text.

In [ ]:
def classify_xlsx_columns(headers: list[str], data_rows: list[tuple], config: Config = CONFIG) -> dict[str, str]:
    columns = {header: [] for header in headers}
    for row in data_rows:
        for header, value in zip(headers, row):
            columns[header].append(value)

    classification = {}
    for header, values in columns.items():
        non_empty = [str(v).strip() for v in values if v is not None and str(v).strip()]
        avg_word_count = (
            sum(len(v.split()) for v in non_empty) / len(non_empty)
            if non_empty else 0
        )

        looks_like_id = header.strip().lower() in config.xlsx_id_like_names
        looks_short = avg_word_count < config.xlsx_semantic_min_avg_words

        classification[header] = "metadata" if (looks_like_id or looks_short) else "semantic"

    return classification

In [ ]:
def read_xlsx_rows(file_path: Path) -> list[tuple[str, list[str], list[tuple]]]:
    workbook = load_workbook(file_path, read_only=True, data_only=True)
    sheets = []

    for sheet in workbook.worksheets:
        rows = list(sheet.iter_rows(values_only=True))
        if not rows:
            continue

        headers = [
            str(value).strip() if value is not None else f"column_{i}"
            for i, value in enumerate(rows[0])
        ]
        sheets.append((sheet.title, headers, rows[1:]))

    workbook.close()
    return sheets


column_preview = []
for file_path in CONFIG.raw_dir.glob("*.xlsx"):
    for sheet_title, headers, data_rows in read_xlsx_rows(file_path):
        roles = classify_xlsx_columns(headers, data_rows)
        for header, role in roles.items():
            column_preview.append({"file": file_path.name, "sheet": sheet_title, "column": header, "role": role})

pd.DataFrame(column_preview)

In [ ]:
def extract_xlsx(file_path: Path, config: Config = CONFIG) -> list[dict]:
    documents = []

    for sheet_title, headers, data_rows in read_xlsx_rows(file_path):
        column_roles = classify_xlsx_columns(headers, data_rows, config)
        semantic_headers = [h for h in headers if column_roles[h] == "semantic"]
        metadata_headers = [h for h in headers if column_roles[h] == "metadata"]

        logger.info(f"{file_path.name} [{sheet_title}]: semantic={semantic_headers} metadata={metadata_headers}")

        for row_number, row in enumerate(data_rows, start=2):
            row_dict = dict(zip(headers, row))

            semantic_values = [
                str(row_dict[h]).strip()
                for h in semantic_headers
                if row_dict.get(h) is not None and str(row_dict[h]).strip()
            ]
            if not semantic_values:
                continue

            text = " | ".join(semantic_values)

            row_metadata = {"sheet": sheet_title, "row": row_number}
            for h in metadata_headers:
                if row_dict.get(h) is not None:
                    row_metadata[h] = row_dict[h]

            documents.append(
                make_record(
                    document=file_path.name,
                    source_type="xlsx",
                    location=f"{sheet_title}!row_{row_number}",
                    text=text,
                    metadata=row_metadata,
                )
            )

    return documents


EXTRACTORS = {".pdf": extract_pdf, ".docx": extract_docx, ".xlsx": extract_xlsx}


def ingest_file(file_path: Path) -> list[dict]:
    extractor = EXTRACTORS.get(file_path.suffix.lower())
    if extractor is None:
        raise ValueError(f"Unsupported file format: {file_path.suffix}")
    return extractor(file_path)

## 5. Run Ingestion

In [ ]:
def run_ingestion(config: Config = CONFIG) -> list[dict]:
    all_documents = []

    for file_path in sorted(config.raw_dir.iterdir()):
        if file_path.suffix.lower() not in config.supported_extensions:
            continue
        try:
            extracted = ingest_file(file_path)
            all_documents.extend(extracted)
            logger.info(f"{file_path.name}: {len(extracted)} records")
        except Exception:
            logger.exception(f"Failed to ingest {file_path.name}, skipping.")

    logger.info(f"Total records: {len(all_documents)}")
    return all_documents


def save_documents(documents: list[dict], config: Config = CONFIG, filename: str = "ingested_records.json") -> Path:
    out_path = config.processed_dir / filename
    out_path.write_text(json.dumps(documents, indent=2, ensure_ascii=False), encoding="utf-8")
    logger.info(f"Saved {len(documents)} records to {out_path}")
    return out_path


def load_documents(config: Config = CONFIG, filename: str = "ingested_records.json") -> list[dict]:
    """Reload records from disk instead of re-running extraction — for picking
    this notebook back up in a fresh kernel without re-parsing every file."""
    path = config.processed_dir / filename
    return json.loads(path.read_text(encoding="utf-8"))


documents = run_ingestion()
save_documents(documents)

### Check the outcome: ingestion & cleaning

In [ ]:
by_format = Counter(d["source_type"] for d in documents)
empty_docs = [d for d in documents if not d["text"]]
short_docs = [d for d in documents if d["word_count"] < CONFIG.short_page_word_threshold]

print(f"Total records: {len(documents)}")
print(f"By format: {dict(by_format)}")
print(f"Empty records: {len(empty_docs)}")
print(f"Suspiciously short records (< {CONFIG.short_page_word_threshold} words): {len(short_docs)}")

# One concrete before/after so cleaning is visibly doing something, not just trusted blindly
sample_raw_source = next(d for d in documents if d["source_type"] == "docx")
print()
print("Cleaning sanity check (whitespace/newline normalization is invisible unless you look for it):")
print("cleaned text repr:", repr(sample_raw_source["text"][:120]))

## 6. Markdown Conversion

DOCX and PDF both get converted to a single Markdown string per document, using whatever heading signal each
format actually has — `style` for DOCX, `font_size`/`is_bold` for PDF, both with the same numbered-heading
fallback (`"3.2 Related Work"` recognized even without a styled/oversized heading). Once both are Markdown,
one splitter handles section-splitting for both — no more `build_docx_sections` **and**
`build_pdf_sections` as two parallel functions.

In [ ]:
def detect_docx_structure(record: dict) -> int | None:
    """Heading level for a DOCX record, or None if it's body text."""
    style = record.get("metadata", {}).get("style", "")

    if style == "Heading 1":
        return 1
    if style == "Heading 2":
        return 2

    text = record.get("text", "").strip()
    match = re.match(r"^(\d+(?:\.\d+)*)\s+.+$", text)
    if match:
        return match.group(1).count(".") + 1

    return None


def docx_records_to_markdown(records: list[dict]) -> str:
    lines = []
    for record in records:
        level = detect_docx_structure(record)
        text = record["text"]
        if level is not None:
            lines.append(f"{'#' * min(level, 6)} {text}")
        else:
            lines.append(text)
    return "\n\n".join(lines)

In [ ]:
def compute_body_font_size(lines: list[dict]) -> float:
    """Most common font size in the document — the body-text baseline headings
    are measured against."""
    if not lines:
        return 0
    return Counter(l["metadata"]["font_size"] for l in lines).most_common(1)[0][0]


def detect_pdf_heading_level(record: dict, body_font_size: float, config: Config = CONFIG) -> int | None:
    """Heading level for a PDF line, or None if it's body text.

    Primary signal: font size relative to the body-text baseline (bigger => higher-level
    heading — the measured equivalent of DOCX's Heading 1/2 styles).
    Fallback signal: a numbered heading, same fallback used for DOCX.
    """
    font_size = record["metadata"]["font_size"]
    is_bold = record["metadata"]["is_bold"]
    size_ratio = font_size / body_font_size if body_font_size else 1.0

    if size_ratio >= config.pdf_heading_size_ratio_h1:
        return 1
    if size_ratio >= config.pdf_heading_size_ratio_h2:
        return 2

    text = record.get("text", "").strip()
    match = re.match(r"^(\d+(?:\.\d+)*)\s+.+$", text)
    if match and (is_bold or size_ratio >= 1.0):
        return match.group(1).count(".") + 1

    return None


def pdf_records_to_markdown(records: list[dict], config: Config = CONFIG) -> str:
    body_font_size = compute_body_font_size(records)

    lines = []
    for record in records:
        level = detect_pdf_heading_level(record, body_font_size, config)
        text = record["text"]
        if level is not None:
            lines.append(f"{'#' * min(level, 6)} {text}")
        else:
            lines.append(text)
    return "\n\n".join(lines)

In [ ]:
docx_records = [d for d in documents if d["source_type"] == "docx"]
pdf_records = [d for d in documents if d["source_type"] == "pdf"]

docx_markdown = {}
for document in {d["document"] for d in docx_records}:
    records = [d for d in docx_records if d["document"] == document]
    docx_markdown[document] = docx_records_to_markdown(records)

pdf_markdown = {}
for document in {d["document"] for d in pdf_records}:
    records = [d for d in pdf_records if d["document"] == document]
    pdf_markdown[document] = pdf_records_to_markdown(records)

print(f"DOCX documents converted: {len(docx_markdown)}")
print(f"PDF documents converted: {len(pdf_markdown)}")

### Check the outcome: markdown conversion

In [ ]:
if docx_markdown:
    sample_doc = next(iter(docx_markdown))
    print(f"--- {sample_doc} (first 600 chars) ---")
    print(docx_markdown[sample_doc][:600])
    print()
    print("Heading lines found:", len(re.findall(r'^#+\s', docx_markdown[sample_doc], flags=re.MULTILINE)))

if pdf_markdown:
    sample_doc = next(iter(pdf_markdown))
    print()
    print(f"--- {sample_doc} (first 600 chars) ---")
    print(pdf_markdown[sample_doc][:600])
    print()
    print("Heading lines found:", len(re.findall(r'^#+\s', pdf_markdown[sample_doc], flags=re.MULTILINE)))

## 7. Section Splitting

One `MarkdownHeaderTextSplitter`, shared by DOCX and PDF, replaces the two hand-written
`build_docx_sections`/`build_pdf_sections` functions from the last notebook — both formats produce Markdown
now, so both can go through the same splitter.

In [ ]:
@dataclass
class StructuralUnit:
    document: str
    source_type: str
    title: str | None
    level: int | None
    text: str
    metadata: dict[str, Any] = field(default_factory=dict)


_markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=list(CONFIG.markdown_headers),
    strip_headers=False,
)


def markdown_to_structural_units(markdown_text: str, document: str, source_type: str) -> list[StructuralUnit]:
    sections = _markdown_splitter.split_text(markdown_text)

    units = []
    for section in sections:
        # section.metadata is keyed by the header *name* ("h1", "h2", ...), not
        # the markdown prefix ("#", "##", ...) — easy to mix up since CONFIG
        # stores (prefix, name) pairs.
        header_keys = [name for _, name in CONFIG.markdown_headers]
        present_headers = [(key, section.metadata[key]) for key in header_keys if key in section.metadata]

        title = present_headers[-1][1] if present_headers else None
        level = len(present_headers) if present_headers else None

        units.append(
            StructuralUnit(
                document=document,
                source_type=source_type,
                title=title,
                level=level,
                text=section.page_content,
                metadata={"section_path": [h for _, h in present_headers]},
            )
        )

    return units

In [ ]:
all_docx_units = []
for document, markdown_text in docx_markdown.items():
    all_docx_units.extend(markdown_to_structural_units(markdown_text, document, "docx"))

all_pdf_units = []
for document, markdown_text in pdf_markdown.items():
    all_pdf_units.extend(markdown_to_structural_units(markdown_text, document, "pdf"))

print(f"DOCX structural units: {len(all_docx_units)}")
print(f"PDF structural units:  {len(all_pdf_units)}")

### Check the outcome: section splitting

In [ ]:
for label, units in [("DOCX", all_docx_units), ("PDF", all_pdf_units)]:
    print(f"--- {label} ---")
    for unit in units[:3]:
        print(f"title={unit.title!r} level={unit.level} path={unit.metadata['section_path']} chars={len(unit.text)}")
    print()

## 8. Chunking

### 8.1 XLSX — direct row-to-chunk

No section splitting, no semantic chunking — a row is already one atomic semantic unit.

In [ ]:
@dataclass
class Chunk:
    chunk_id: str
    document: str
    source_type: str
    text: str
    metadata: dict[str, Any] = field(default_factory=dict)


def xlsx_records_to_chunks(records: list[dict]) -> list[Chunk]:
    chunks = []
    for index, record in enumerate(records):
        chunks.append(
            Chunk(
                chunk_id=f"row_{index}",
                document=record["document"],
                source_type="xlsx",
                text=record["text"],
                metadata={k: v for k, v in record["metadata"].items()},
            )
        )
    return chunks


xlsx_records = [d for d in documents if d["source_type"] == "xlsx"]
all_xlsx_chunks = xlsx_records_to_chunks(xlsx_records)

print(f"XLSX records: {len(xlsx_records)}")
print(f"XLSX chunks (1:1 with records): {len(all_xlsx_chunks)}")

### 8.2 DOCX & PDF — semantic chunking, with a fixed-size fallback

Splits a section wherever the meaning shifts between sentences (embedding similarity drops), instead of
cutting at a fixed character count regardless of what's mid-thought at that point. Falls back to the
fixed-size sliding-window chunker (same one, bug-fixed, from the last notebook) in three cases: the embedding
model isn't available, the section is too short to meaningfully group, or a semantic chunk still comes out
oversized.

In [ ]:
_embedding_model = None
if _SENTENCE_TRANSFORMERS_AVAILABLE:
    try:
        _embedding_model = SentenceTransformer(CONFIG.embedding_model_name)
        logger.info(f"Loaded embedding model: {CONFIG.embedding_model_name}")
    except Exception:
        logger.exception("Could not load embedding model — falling back to fixed-size chunking only.")
        _embedding_model = None
else:
    logger.warning("sentence-transformers not installed — falling back to fixed-size chunking only.")


def split_into_sentences(text: str) -> list[str]:
    raw = re.split(r'(?<=[.!?])\s+(?=[A-Z0-9])', text.strip())
    return [s.strip() for s in raw if s.strip()]


def semantic_breakpoints(embeddings: np.ndarray, percentile: float) -> list[int]:
    """Indices after which a section should be split — where the cosine distance
    between consecutive sentence embeddings exceeds the given percentile of all
    distances in this section (a bigger jump = bigger topic shift)."""
    a, b = embeddings[:-1], embeddings[1:]
    similarities = (a * b).sum(axis=1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + 1e-8)
    distances = 1 - similarities

    if len(distances) == 0:
        return []

    threshold = np.percentile(distances, percentile)
    return [i for i, d in enumerate(distances) if d > threshold]

In [ ]:
def chunk_structural_unit(unit: StructuralUnit, chunk_size: int = 1500, overlap: int = 200) -> list[dict]:
    """Fixed-size sliding-window chunker — the fallback path."""
    text = unit.text.strip()

    if not text:
        return []

    if len(text) <= chunk_size:
        return [{
            "document": unit.document,
            "source_type": unit.source_type,
            "text": text,
            "metadata": {**unit.metadata, "section_title": unit.title, "section_level": unit.level, "chunk_index": 0},
        }]

    chunks = []
    start = 0
    chunk_index = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))

        if end < len(text):
            boundary = text.rfind(" ", start, end)
            if boundary > start:
                end = boundary

        chunk_text = text[start:end].strip()
        if chunk_text:
            chunks.append({
                "document": unit.document,
                "source_type": unit.source_type,
                "text": chunk_text,
                "metadata": {**unit.metadata, "section_title": unit.title, "section_level": unit.level, "chunk_index": chunk_index},
            })
            chunk_index += 1

        if end >= len(text):
            break

        next_start = max(end - overlap, start + 1)
        boundary = text.find(" ", next_start, end)
        start = boundary + 1 if boundary != -1 else next_start

    return chunks


def semantic_chunk_unit(unit: StructuralUnit, config: Config = CONFIG) -> list[dict]:
    text = unit.text.strip()
    if not text:
        return []

    sentences = split_into_sentences(text)

    # Too short to meaningfully group, or no embedding model available — fixed-size fallback.
    if _embedding_model is None or len(sentences) < config.semantic_chunk_min_sentences:
        return chunk_structural_unit(unit, chunk_size=config.semantic_chunk_max_chars, overlap=config.fixed_chunk_overlap)

    embeddings = _embedding_model.encode(sentences)
    breakpoints = semantic_breakpoints(embeddings, config.semantic_chunk_percentile)

    chunk_texts, start = [], 0
    for bp in breakpoints:
        chunk_texts.append(" ".join(sentences[start:bp + 1]))
        start = bp + 1
    chunk_texts.append(" ".join(sentences[start:]))
    chunk_texts = [c.strip() for c in chunk_texts if c.strip()]

    chunk_dicts = []
    for chunk_index, chunk_text in enumerate(chunk_texts):
        if len(chunk_text) > config.semantic_chunk_max_chars:
            # oversized semantic chunk — split further with the fixed-size chunker
            oversized_unit = StructuralUnit(unit.document, unit.source_type, unit.title, unit.level, chunk_text, unit.metadata)
            chunk_dicts.extend(chunk_structural_unit(oversized_unit, chunk_size=config.semantic_chunk_max_chars, overlap=config.fixed_chunk_overlap))
        else:
            chunk_dicts.append({
                "document": unit.document,
                "source_type": unit.source_type,
                "text": chunk_text,
                "metadata": {**unit.metadata, "section_title": unit.title, "section_level": unit.level, "chunk_index": chunk_index},
            })

    return chunk_dicts


def chunk_dicts_to_objects(chunk_dicts: list[dict], unit_index: int = 0) -> list[Chunk]:
    return [
        Chunk(
            chunk_id=f"unit_{unit_index}::chunk_{chunk_index}",
            document=item["document"],
            source_type=item["source_type"],
            text=item["text"],
            metadata=item["metadata"],
        )
        for chunk_index, item in enumerate(chunk_dicts)
    ]

In [ ]:
all_docx_chunks = []
for unit_index, unit in enumerate(all_docx_units):
    chunk_dicts = semantic_chunk_unit(unit)
    all_docx_chunks.extend(chunk_dicts_to_objects(chunk_dicts, unit_index=unit_index))

all_pdf_chunks = []
for unit_index, unit in enumerate(all_pdf_units):
    chunk_dicts = semantic_chunk_unit(unit)
    all_pdf_chunks.extend(chunk_dicts_to_objects(chunk_dicts, unit_index=unit_index))

print(f"DOCX chunks: {len(all_docx_chunks)}")
print(f"PDF chunks:  {len(all_pdf_chunks)}")
print(f"XLSX chunks: {len(all_xlsx_chunks)}")

### Check the outcome: chunking

In [ ]:
all_chunks = all_docx_chunks + all_pdf_chunks + all_xlsx_chunks

for label, chunks in [("DOCX", all_docx_chunks), ("PDF", all_pdf_chunks), ("XLSX", all_xlsx_chunks)]:
    if not chunks:
        print(f"{label}: no chunks")
        continue
    lengths = [len(c.text) for c in chunks]
    print(f"{label}: {len(chunks)} chunks | min={min(lengths)} max={max(lengths)} avg={sum(lengths)/len(lengths):.0f}")

oversized = [c for c in all_chunks if c.source_type != "xlsx" and len(c.text) > CONFIG.semantic_chunk_max_chars]
missing_provenance = [c for c in all_chunks if not c.document or not c.source_type]

print()
print(f"Total chunks across all formats: {len(all_chunks)}")
print(f"Oversized chunks (docx/pdf > {CONFIG.semantic_chunk_max_chars} chars): {len(oversized)}")
print(f"Chunks missing provenance: {len(missing_provenance)}")

print()
print("Sample chunk (docx or pdf, if any):")
sample = next((c for c in all_chunks if c.source_type in ("docx", "pdf")), None)
if sample:
    print("section:", sample.metadata.get("section_title"))
    print("text:", sample.text[:300])

## 9. Save Chunks

In [ ]:
def save_chunks(chunks: list[Chunk], config: Config = CONFIG, filename: str = "chunks.json") -> Path:
    out_path = config.processed_dir / filename
    payload = [
        {"chunk_id": c.chunk_id, "document": c.document, "source_type": c.source_type, "text": c.text, "metadata": c.metadata}
        for c in chunks
    ]
    out_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    logger.info(f"Saved {len(chunks)} chunks to {out_path}")
    return out_path


save_chunks(all_chunks)

## Next up: Deduplication

Hash each chunk, check the hash against what's already been embedded, and only send new/changed chunks to
the embedding step — sitting right here, between this chunking output and Qdrant. Say the word whenever
you're ready to walk through it.